<div class="alert alert-info" role="alert" style="padding:20px; margin-bottom:16px;">
  <div style="text-align:center;">
    <h1 style="margin:0;">Exploratory Data Analysis</h1>
    <div style="font-size:18px; margin-top:4px;">Amazing International Airlines Inc.</div>
    <hr style="margin:12px auto; width:220px;">
    <div style="font-size:14px; color:#6c757d;">Group 92 • Notebook • 2025/2026</div>
  </div>
</div>


This Project was done by:


Student Name    -   Mehmet Karaca;
student id      -   20250344;
contact email   -   20250344@novaims.unl.pt

Student Name    -   Duarte Gomes;
student id      -   20250017;
contact email   -   20250017@novaims.unl.pt

Student Name    -   Esra Salhi
student id      -   20250537
contact email   -   20250537@novaims.unl.pt

## Table of Contents



Add some contents here



# 1 Introduction <a id="introduction"></a>


## Context and Metadata <a id="context-and-metadata"></a>


CustomerDB 

`Unnamed` - Column without a name or meaning <br>
`Loyalty#` - Loyalty program identifier for the customer <br>
`First Name` - Customer's first name <br>
`Last Name` - Customer's last name <br>
`Customer Name` - Full name of the customer <br>
`Country` - Country where the customer resides <br>
`Province or State` - Province or state of the customer's residence <br>
`City` - City of the customer's residence <br>
`Latitude` - Latitude coordinate of the customer's location <br>
`Longitude` - Longitude coordinate of the customer's location <br>
`Postal code` - Postal code of the customer's address <br>
`Gender` - Gender of the customer <br>
`Education` - Education level of the customer <br>
`Location Code` - Code representing the customer's location <br>
`Income` - Income level of the customer <br>
`Marital Status` - Marital status of the customer <br>
`LoyaltyStatus` - Status of the customer's loyalty program <br>
`EnrollmentDateOpening` - Date when the customer enrolled in the loyalty program <br>
`CancellationDate` - Date when the customer canceled their loyalty program <br>
`Customer Lifetime Value` - Total value of the customer over their lifetime <br>
`EnrollmentType` - Type of enrollment in the loyalty program <br>


FlightsDB

`Loyalty#` - Loyalty program identifier for the customer <br>
`Year` - Year of the flight activity <br>
`Month` - Month of the flight activity <br>
`YearMonthDate` - Date representing the year and month of the flight activity <br>
`NumFlights` - Number of flights taken by the customer <br>
`NumFlightsWithCompanions` - Number of flights taken with companions <br>
`DistanceKM` - Total distance traveled in kilometers <br>
`PointsAccumulated` - Points accumulated by the customer <br>
`PointsRedeemed` - Points redeemed by the customer <br>
`DollarCostPointsRedeemed` - Dollar cost of the points redeemed <br>


## 1.1. Importing Libraries <a id="importing-libraries"></a>


In [17]:
# --- Standard Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import matplotlib
import math
import matplotlib as mpl
from cycler import cycler
import geopandas as gpd
from matplotlib.lines import Line2D
from sklearn.impute import KNNImputer



In [18]:
# Plot style configuration (unified look)
# Keeps logic intact; only global appearance defaults are set.

# Accessible, consistent palette
_palette = sns.color_palette("Set1", 10)
_palette_hex = _palette.as_hex()

# Seaborn / Matplotlib defaults
sns.set_theme(style="whitegrid", context="notebook", palette=_palette)
mpl.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.titleweight': 'semibold',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.color': '#EAEAEA',
    'grid.linestyle': '-',
    'grid.alpha': 0.6,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.borderpad': 0.4,
    'legend.loc': 'best',
    'lines.linewidth': 2.0,
    'axes.prop_cycle': cycler('color', _palette_hex),
})


def make_palette_map(categories, palette_name="Set1", n_colors=None, as_hex=False):
    """
    Simple: returns (palette_map, palette_list).
    - categories: iterable (keeps first-seen order, coerces to str)
    - palette_name: seaborn palette name
    - n_colors: optional override (defaults to max(len(categories), 3))
    - as_hex: if True, return palette_map with hex strings
    """
    cats = [str(c) for c in dict.fromkeys(categories)]          # unique keep-order
    n_colors = n_colors or max(len(cats), 3)
    palette = sns.color_palette(palette_name, n_colors)
    palette_map = {cat: palette[i % len(palette)] for i, cat in enumerate(cats)}
    if as_hex:
        return {k: matplotlib.colors.to_hex(v) for k, v in palette_map.items()}, palette
    return palette_map, palette



# Plotly defaults (to align with seaborn/matplotlib)
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = _palette_hex
px.defaults.color_continuous_scale = 'Cividis'
px.defaults.width = 900
px.defaults.height = 500


## 1.2. Loading and Reading Data <a id="loading-and-reading-data"></a>

In [19]:
# directory with raw CSV files
data_dir = Path("../data/raw")

# list all CSV files
csv_files = list(data_dir.glob("*.csv"))
print(f"Found CSV files: {[f.name for f in csv_files]}")


Found CSV files: ['DM_AIAI_CustomerDB.csv', 'DM_AIAI_Metadata.csv', 'DM_AIAI_FlightsDB.csv']


In [20]:
# specify the files we want to load

customers_file = data_dir / "DM_AIAI_CustomerDB.csv"
flights_file   = data_dir / "DM_AIAI_FlightsDB.csv"

# load them with pandas
customers = pd.read_csv(customers_file)
flights   = pd.read_csv(flights_file)

print("Customers shape:", customers.shape)
print("Flights shape:", flights.shape)

Customers shape: (16921, 21)
Flights shape: (608436, 10)


## 1.3. Brief Preliminary Analysis <a id="brief-preliminary-analysis"></a>

In [21]:
customers.columns

Index(['Unnamed: 0', 'Loyalty#', 'First Name', 'Last Name', 'Customer Name',
       'Country', 'Province or State', 'City', 'Latitude', 'Longitude',
       'Postal code', 'Gender', 'Education', 'Location Code', 'Income',
       'Marital Status', 'LoyaltyStatus', 'EnrollmentDateOpening',
       'CancellationDate', 'Customer Lifetime Value', 'EnrollmentType'],
      dtype='object')

In [22]:
customers.info

<bound method DataFrame.info of        Unnamed: 0  Loyalty# First Name    Last Name        Customer Name  \
0               0    480934    Cecilia  Householder  Cecilia Householder   
1               1    549612      Dayle        Menez          Dayle Menez   
2               2    429460     Necole       Hannon        Necole Hannon   
3               3    608370      Queen        Hagee          Queen Hagee   
4               4    530508     Claire      Latting       Claire Latting   
...           ...       ...        ...          ...                  ...   
16916          15    100012      Ethan     Thompson       Ethan Thompson   
16917          16    100013      Layla        Young          Layla Young   
16918          17    100014     Amelia      Bennett       Amelia Bennett   
16919          18    100015   Benjamin       Wilson      Benjamin Wilson   
16920          19    100016       Emma       Martin          Emma Martin   

      Country Province or State          City   Latitud

In [23]:
customers.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,16921.0,8440.023639,4884.775439,0.000000,4210.000000,8440.000000,12670.000000,16900.000000
Loyalty#,16921.0,550197.393771,259251.503597,100011.000000,326823.000000,550896.000000,772438.000000,999999.000000
Latitude,16921.0,47.174500,3.307971,42.984924,44.231171,46.087818,49.282730,60.721188
Longitude,16921.0,-91.814768,22.242429,-135.056840,-120.237660,-79.383186,-74.596184,-52.712578
Income,16901.0,37758.038400,30368.992499,0.000000,0.000000,34161.000000,62396.000000,99981.000000
Customer Lifetime Value,16901.0,7990.460188,6863.173093,1898.010000,3979.720000,5780.180000,8945.690000,83325.380000


In [24]:
customers.describe(include='object').T

,count,unique,top,freq
First Name,16921,4941,Deon,13
Last Name,16921,15404,Salberg,4
Customer Name,16921,16921,Cecilia Householder,1
Country,16921,1,Canada,16921
Province or State,16921,11,Ontario,5468
City,16921,29,Toronto,3390
Postal code,16921,75,V6E 3D9,917
Gender,16921,2,female,8497
Education,16921,5,Bachelor,10586
Location Code,16921,3,Suburban,5716


In [25]:
flights.columns

Index(['Loyalty#', 'Year', 'Month', 'YearMonthDate', 'NumFlights',
       'NumFlightsWithCompanions', 'DistanceKM', 'PointsAccumulated',
       'PointsRedeemed', 'DollarCostPointsRedeemed'],
      dtype='object')

In [26]:
flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608436 entries, 0 to 608435
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Loyalty#                  608436 non-null  int64  
 1   Year                      608436 non-null  int64  
 2   Month                     608436 non-null  int64  
 3   YearMonthDate             608436 non-null  object 
 4   NumFlights                608436 non-null  float64
 5   NumFlightsWithCompanions  608436 non-null  float64
 6   DistanceKM                608436 non-null  float64
 7   PointsAccumulated         608436 non-null  float64
 8   PointsRedeemed            608436 non-null  float64
 9   DollarCostPointsRedeemed  608436 non-null  float64
dtypes: float64(6), int64(3), object(1)
memory usage: 46.4+ MB


In [27]:
flights.describe().T

,count,mean,std,min,25%,50%,75%,max
Loyalty#,608436.0,550037.873084,258935.180575,100018.0,326961.00,550834.000,772194.0000,999986.0
Year,608436.0,2020.000000,0.816497,2019.0,2019.00,2020.000,2021.0000,2021.0
Month,608436.0,6.500000,3.452055,1.0,3.75,6.500,9.2500,12.0
NumFlights,608436.0,3.908107,5.057889,0.0,0.00,0.000,7.2000,21.0
NumFlightsWithCompanions,608436.0,0.983944,2.003785,0.0,0.00,0.000,0.9000,11.0
DistanceKM,608436.0,7939.341419,10260.421873,0.0,0.00,856.400,15338.1750,42040.0
PointsAccumulated,608436.0,793.777781,1025.918521,0.0,0.00,85.275,1533.7125,4204.0
PointsRedeemed,608436.0,235.251678,983.233374,0.0,0.00,0.000,0.0000,7496.0
DollarCostPointsRedeemed,608436.0,2.324835,9.725168,0.0,0.00,0.000,0.0000,74.0


In [28]:
flights.describe(include='object').T

,count,unique,top,freq
YearMonthDate,608436,36,12/1/2021,16901


Remove the column "Unnamed: 0" because it is not needed and has no context in the data.

In [29]:
if "Unnamed: 0" in customers.columns:
	customers.drop(columns=["Unnamed: 0"], inplace=True)


# Check for the column again
print("Columns after dropping 'Unnamed: 0':", customers.columns.tolist())

Columns after dropping 'Unnamed: 0': ['Loyalty#', 'First Name', 'Last Name', 'Customer Name', 'Country', 'Province or State', 'City', 'Latitude', 'Longitude', 'Postal code', 'Gender', 'Education', 'Location Code', 'Income', 'Marital Status', 'LoyaltyStatus', 'EnrollmentDateOpening', 'CancellationDate', 'Customer Lifetime Value', 'EnrollmentType']


# 2. Missing Values and Data Validity Checks  <a class="anchor" id="data-validity-checks"></a>

We only check smissing values , because in the first deliverable. we are only allowed to notice them and do little cleaning .but we need to notice them .

In [30]:
# Converting the datatype of categorical features from object to category

flights['YearMonthDate'] = pd.to_datetime(flights['YearMonthDate'], format='%m/%d/%Y')


# Convert NumFlights and NumFlightsWithCompanions to integer type (they are float by default)
flights["NumFlights"] = flights["NumFlights"].astype(int)
flights["NumFlightsWithCompanions"] = flights["NumFlightsWithCompanions"].astype(int)

In [31]:
# Check the datatype of YearMonthDate column
flights['YearMonthDate'].dtype

dtype('<M8[ns]')

## 2.1. Missing Values <a id="missing-values"></a>

Some text to describe what we are doing here.

In [32]:
# Check for missing values
print("Missing values in CustomerDB:")
display(customers.isna().sum())

print("\nMissing values in FlightsDB:")
display(flights.isna().sum())


Missing values in CustomerDB:


Loyalty#                       0
First Name                     0
Last Name                      0
Customer Name                  0
Country                        0
Province or State              0
City                           0
Latitude                       0
Longitude                      0
Postal code                    0
Gender                         0
Education                      0
Location Code                  0
Income                        20
Marital Status                 0
LoyaltyStatus                  0
EnrollmentDateOpening          0
CancellationDate           14611
Customer Lifetime Value       20
EnrollmentType                 0
dtype: int64


Missing values in FlightsDB:


Loyalty#                    0
Year                        0
Month                       0
YearMonthDate               0
NumFlights                  0
NumFlightsWithCompanions    0
DistanceKM                  0
PointsAccumulated           0
PointsRedeemed              0
DollarCostPointsRedeemed    0
dtype: int64

In [33]:
missing_ratio = customers.isna().mean().sort_values(ascending=False)
print((missing_ratio * 100).round(2))

CancellationDate           86.35
Customer Lifetime Value     0.12
Income                      0.12
Loyalty#                    0.00
First Name                  0.00
EnrollmentDateOpening       0.00
LoyaltyStatus               0.00
Marital Status              0.00
Location Code               0.00
Education                   0.00
Gender                      0.00
Postal code                 0.00
Longitude                   0.00
Latitude                    0.00
City                        0.00
Province or State           0.00
Country                     0.00
Customer Name               0.00
Last Name                   0.00
EnrollmentType              0.00
dtype: float64


The flights dataset is clean. The customers dataset has missing values in the columns "CancellationDate", "Customer Lifetime Value" and "Income". We start with analyzing the CancellationDate column. We expect that, these missing values correspond to customers who have not cancelled their subscription. We will check this assumption by looking at some entries with missing CancellationDate values.

In [34]:
# select first 10 rows where CancellationDate is NaN
active_customers = customers[customers["CancellationDate"].isna()].head(10)
active_customers

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,female,Bachelor,Urban,70146.0,Married,Star,2/15/2019,NaN,3839.14,Standard
1,549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,male,College,Rural,0.0,Divorced,Star,3/9/2019,NaN,3839.61,Standard
3,608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,2/17/2016,NaN,3839.75,Standard
4,530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,male,Bachelor,Suburban,97832.0,Married,Star,10/25/2017,NaN,3842.79,2021 Promotion
5,193662,Leatrice,Hanlin,Leatrice Hanlin,Canada,Yukon,Whitehorse,60.721188,-135.056840,Y2K 6R0,male,Bachelor,Rural,26262.0,Married,Star,5/7/2015,NaN,3844.57,Standard
6,927943,Hue,Sellner,Hue Sellner,Canada,Ontario,Toronto,43.653225,-79.383186,P5S 6R4,female,College,Urban,0.0,Single,Star,6/9/2017,NaN,3857.95,Standard
7,188893,Nakia,Cash,Nakia Cash,Canada,Ontario,Trenton,44.101128,-77.576309,K8V 4B2,male,Bachelor,Suburban,93272.0,Married,Star,12/8/2019,NaN,3861.49,Standard
8,852392,Arlene,Conterras,Arlene Conterras,Canada,Quebec,Montreal,45.501690,-73.567253,H2Y 2W2,female,Bachelor,Suburban,93272.0,Married,Star,5/30/2018,NaN,3861.49,Standard
9,866307,Dustin,Recine,Dustin Recine,Canada,Ontario,Toronto,43.653225,-79.383186,M8Y 4K8,male,Bachelor,Suburban,93272.0,Married,Star,10/14/2019,NaN,3861.49,Standard
10,932823,Jeremy,Dickason,Jeremy Dickason,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,female,Bachelor,Suburban,47199.0,Married,Star,3/17/2018,NaN,3863.31,Standard


The Entries seem to look fine. The expectation seems to be correct. We will add a new column in the Data Engineering section to indicate whether a customer is active or not based on the CancellationDate column.

Next we check the missing values in the "Customer Lifetime Value" and "Income" columns. 

In [35]:
# rows where Income is missing
missing_income = customers[customers["Income"].isna()]
print("Rows with missing Income:", missing_income.shape[0])
display(missing_income.head(5))

Rows with missing Income: 20


,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
16901,999987,Layla,Murphy,Layla Murphy,Canada,New Brunswick,Fredericton,46.029263,-66.565150,R4H 2Y2,female,Bachelor,Urban,NaN,Single,Star,3/7/2017,3/7/2017,NaN,Standard
16902,999988,Jana,Parker,Jana Parker,Canada,Quebec,Montreal,45.573672,-73.523012,N6B 1N3,male,College,Rural,NaN,Single,Star,8/22/2017,8/22/2017,NaN,Standard
16903,999989,Ethan,Parker,Ethan Parker,Canada,Ontario,Trenton,44.075379,-77.550375,P8F 5C8,male,College,Rural,NaN,Married,Star,9/12/2015,9/12/2015,NaN,Standard
16904,999990,Ryan,Anderson,Ryan Anderson,Canada,New Brunswick,Moncton,46.106617,-64.714267,B6P 6D0,female,College,Rural,NaN,Married,Star,6/10/2019,6/10/2019,NaN,Standard
16905,999991,Olivia,Cote,Olivia Cote,Canada,New Brunswick,Fredericton,45.950000,-66.652437,X3W 5N2,female,College,Suburban,NaN,Married,Star,7/20/2019,7/20/2019,NaN,Standard


In [36]:
# rows where Customer Lifetime Value is missing
missing_clv = customers[customers["Customer Lifetime Value"].isna()]
print("Rows with missing Customer Lifetime Value:", missing_clv.shape[0])
display(missing_clv.head(5))

Rows with missing Customer Lifetime Value: 20


,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
16901,999987,Layla,Murphy,Layla Murphy,Canada,New Brunswick,Fredericton,46.029263,-66.565150,R4H 2Y2,female,Bachelor,Urban,NaN,Single,Star,3/7/2017,3/7/2017,NaN,Standard
16902,999988,Jana,Parker,Jana Parker,Canada,Quebec,Montreal,45.573672,-73.523012,N6B 1N3,male,College,Rural,NaN,Single,Star,8/22/2017,8/22/2017,NaN,Standard
16903,999989,Ethan,Parker,Ethan Parker,Canada,Ontario,Trenton,44.075379,-77.550375,P8F 5C8,male,College,Rural,NaN,Married,Star,9/12/2015,9/12/2015,NaN,Standard
16904,999990,Ryan,Anderson,Ryan Anderson,Canada,New Brunswick,Moncton,46.106617,-64.714267,B6P 6D0,female,College,Rural,NaN,Married,Star,6/10/2019,6/10/2019,NaN,Standard
16905,999991,Olivia,Cote,Olivia Cote,Canada,New Brunswick,Fredericton,45.950000,-66.652437,X3W 5N2,female,College,Suburban,NaN,Married,Star,7/20/2019,7/20/2019,NaN,Standard


In [37]:
# Check if the missing values in Income and Customer Lifetime Value overlap
missing_income_indices = set(missing_income.index)
missing_clv_indices = set(missing_clv.index)
overlapping_indices = missing_income_indices.intersection(missing_clv_indices)

print("Rows with missing values in both Income and Customer Lifetime Value:", len(overlapping_indices))

Rows with missing values in both Income and Customer Lifetime Value: 20


The missing values in Income and Customer Lifetime Value do overlap fully. This means that all rows with missing Income also have missing Customer Lifetime Value. This could indicate a correlation between the two columns, or it could be due to data collection issues. Only 20 rows are affected, which is a small fraction of the total dataset. We will delete them, because they are only 20 rows and will not impact the analysis significantly.

In [38]:
# Delete entries with with NaN Income and Customer Lifetime Value
customers = customers.dropna(subset=["Income", "Customer Lifetime Value"])

# Check for remaining missing values
remaining_missing_income = customers[customers["Income"].isna()]
remaining_missing_clv = customers[customers["Customer Lifetime Value"].isna()]
print("Remaining rows with missing Income after deletion:", remaining_missing_income.shape[0])
print("Remaining rows with missing Customer Lifetime Value after deletion:", remaining_missing_clv.shape[0])

Remaining rows with missing Income after deletion: 0
Remaining rows with missing Customer Lifetime Value after deletion: 0


## 2.2. Validity check <a id="validity-check"></a>

We check all the columns for validity of the values.

### 2.2.1 Categorical Features of customers <a id="categorical-features-of-customers"></a>

In [39]:
# List all categorical columns in customers
cat_columns_customers = customers.select_dtypes(include=['object']).columns.tolist()
cat_columns_customers

['First Name',
 'Last Name',
 'Customer Name',
 'Country',
 'Province or State',
 'City',
 'Postal code',
 'Gender',
 'Education',
 'Location Code',
 'Marital Status',
 'LoyaltyStatus',
 'EnrollmentDateOpening',
 'CancellationDate',
 'EnrollmentType']

In [40]:
# Check the categorical columns for unexpected values
for col in ["Gender","Country", "Education", "Marital Status","LoyaltyStatus", "Province or State", "City", "EnrollmentType"]:
    if col in customers.columns:
        print(f"\n{col} categories:")
        print(customers[col].value_counts(dropna=False))


Gender categories:
Gender
female    8486
male      8415
Name: count, dtype: int64

Country categories:
Country
Canada    16901
Name: count, dtype: int64

Education categories:
Education
Bachelor                10578
College                  4273
High School or Below      792
Doctor                    742
Master                    516
Name: count, dtype: int64

Marital Status categories:
Marital Status
Married     9830
Single      4531
Divorced    2540
Name: count, dtype: int64

LoyaltyStatus categories:
LoyaltyStatus
Star      7741
Nova      5722
Aurora    3438
Name: count, dtype: int64

Province or State categories:
Province or State
Ontario                 5462
British Columbia        4413
Quebec                  3307
Alberta                 1006
Manitoba                 676
New Brunswick            647
Nova Scotia              541
Saskatchewan             412
Newfoundland             258
Yukon                    112
Prince Edward Island      67
Name: count, dtype: int64

City categ

In [41]:
# Check for postal codes
print("Unique postal codes:", customers["Postal code"].nunique())
print(customers["Postal code"].value_counts().head(10))

#Check if Postal code has entries with more or less than 7 characters
long_postal_codes = customers[customers["Postal code"].astype(str).str.len() > 7]
print("Postal codes with more than 7 characters:", long_postal_codes.shape[0])
short_postal_codes = customers[customers["Postal code"].astype(str).str.len() < 7]
print("Postal codes with less than 7 characters:", short_postal_codes.shape[0])


Unique postal codes: 55
Postal code
V6E 3D9    917
V5R 1W3    699
V6T 1Y8    586
M2M 7K8    538
V6E 3Z3    538
P1J 8T7    508
H2T 9K8    503
K8V 4B2    490
G1B 3L5    481
H2T 2J6    449
Name: count, dtype: int64
Postal codes with more than 7 characters: 0
Postal codes with less than 7 characters: 0


Check of the unique values in categorical columns to see if they make sense.

- Gender: Only male and female --> valid
- Country: Only Canada --> valid
- Education: Only valid education levels --> valid
- Marital Status: Only valid marital statuses --> valid
- LoyaltyStatus: Only valid loyalty statuses --> valid
- Province or State: Only valid provinces or states in Canada --> valid
- City: Only valid cities in Canada --> valid
- EnrollmentType: Only valid enrollment types --> valid

### 2.2.2 Numerical Features of customers <a id="numerical-features-of-customers"></a>


In [42]:
# Min and Max Values of each column of customers db
numeric_cols = customers.select_dtypes(include=["number"]).columns

min_max = customers[numeric_cols].agg(["min", "max"]).T
min_max.transpose()


,Loyalty#,Latitude,Longitude,Income,Customer Lifetime Value
min,100018.0,42.984924,-135.056840,0.0,1898.01
max,999986.0,60.721188,-52.712578,99981.0,83325.38


- Loyalty#: Is the unique identifier for each customer. Check if unique
- Lattitude and Longitude make sense because it is only data from Canada (Longitutde: -52 to -141, Latitude: 41 to 83)
- Income has no negative values, but needs to be checked for outliers
- Customer Lifetime Value has no negative values, but needs to be checked for outliers
- IsActive makes sense because it is only 0 and 1


Next we check for unique loyalty# 

In [43]:
# Check uniqueness of Loyalty in customers
n_rows = customers.shape[0]
n_unique_ids = customers["Loyalty#"].nunique()

print("Rows in CustomerDB:", n_rows)
print("Unique Loyalty IDs:", n_unique_ids)

if n_rows == n_unique_ids:
    print("Loyalty is unique per row.")
else:
    print ("There are", n_rows - n_unique_ids - 1, "duplicated Loyalty values.")


Rows in CustomerDB: 16901
Unique Loyalty IDs: 16737
There are 163 duplicated Loyalty values.


- There are some duplicated Loyalty values in the customers dataset. We have to check if these are really duplicates or if there are different customers with the same loyalty id. If they are duplicates, we have to delete them.

In [44]:
# List of the inconsistent Loyalty IDs in customers dataset
duplicates_by_name = customers[customers.duplicated(subset=["Loyalty#"], keep=False)].sort_values(by=["Loyalty#"])
duplicates_by_name[['Loyalty#', 'First Name', 'Last Name']]

,Loyalty#,First Name,Last Name
1646,101902,Hans,Schlottmann
2668,101902,Yi,Nesti
15988,106001,Maudie,Hyland
700,106001,Ivette,Peifer
13053,106509,Stacy,Schwebke
...,...,...,...
5038,989528,Sharri,Boughman
9890,990512,Magda,Sopher
14478,990512,Ione,Snowden
6981,992168,Frederick,Samaha


- That means that different customers have the same loyalty id. This is a problem because we cannot identify the customers uniquely and can not map their flights correctly. We check how many entries in flights correspond to these inconsistent loyalty ids.
  

In [45]:
# Count the number of entries in flights corresponding to the inconsistent loyalty ids in total
inconsistent_loyalty_ids = duplicates_by_name['Loyalty#'].unique()
flights_inconsistent = flights[flights['Loyalty#'].isin(inconsistent_loyalty_ids)]
flights_inconsistent_count = flights_inconsistent['Loyalty#'].value_counts()
flights_inconsistent_count.sum()
print("Number of flight entries with inconsistent loyalty IDs:", flights_inconsistent_count.sum())

# Percentage of flights affected by inconsistent loyalty ids
total_flights = flights.shape[0]
affected_percentage = (flights_inconsistent_count.sum() / total_flights) * 100
print(f"Percentage of flights affected by inconsistent loyalty IDs: {affected_percentage:.2f}%")

Number of flight entries with inconsistent loyalty IDs: 11772
Percentage of flights affected by inconsistent loyalty IDs: 1.93%


- One Major problem of these duplicated loyalty ids is, that we cannot link the customers to their flights. There is no reasonable way to differentiate between the two customers with the same loyalty id. So that the even changing the loyalty id of one customer would not help, because we still do not know which flights belong to which customer. We would create a new customer with a new loyalty id, but this customer would have no flights linked to him. So we would loose all the information about the flights of these customers and the other customer would have all the flights linked to him. This would distort the data and make it unusable for analysis.
  
- So for this reason we have to delete all customers with duplicated loyalty ids and all their flights. This is the only way to ensure that the remaining data is valid and can be used for analysis.

- We may lose some customers, but there is no other way to ensure the validity of the data. The percentage of affected customers is 1.93%, which is relatively low compared to the total number of customers, so the impact on the analysis should be minimal.

In [46]:
# Clean the customers and flights datasets by removing entries with duplicated Loyalty IDs
loyalty_ids_to_remove = duplicates_by_name['Loyalty#'].unique()
customers = customers[~customers['Loyalty#'].isin(loyalty_ids_to_remove)]
flights = flights[~flights['Loyalty#'].isin(loyalty_ids_to_remove)]
print("Cleaned Customers shape:", customers.shape)
print("Cleaned Flights shape:", flights.shape)

Cleaned Customers shape: (16574, 20)
Cleaned Flights shape: (596664, 10)


### 2.2.3 Numerical Features of flights <a id="numerical-features-of-flights"></a>


In [47]:
# Min and Max Values of each column of flights db
numeric_cols = flights.select_dtypes(include=["number"]).columns

min_max = flights[numeric_cols].agg(["min", "max"]).T
min_max.transpose()

,Loyalty#,Year,Month,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
min,100018.0,2019.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
max,999986.0,2021.0,12.0,21.0,11.0,42040.0,4204.0,7496.0,74.0


Explanation of the validity check:

- Year is between 2019 and 2021 --> makes sense
- Month is between 1 and 12 --> makes sense
- Day is between 1 and 31 --> makes sense
- NumFlights is between 0 and 21 --> makes sense 
- NumFlightsWithCompanions is between 0 and 11 --> makes sense
- DistanceKM is between 0 and 42040 --> makes sense
- PointsAccumulated is between 0 and 42040 --> makes sense
- PointsRedeemed is between 0 and 7496 --> makes sense
- DollarCostPointsRedeemed is between 0 and 74 --> makes sense

Validity check of YearMonthDate column:

### 2.2.4 Datetime Features of Flights <a id="datetime-features-of-flights"></a>


In [48]:
# Print all the unique values of YearMonthDate column sorted by date and their freuquency
flights['YearMonthDate'].value_counts().sort_index()

YearMonthDate
2019-01-01    16574
2019-02-01    16574
2019-03-01    16574
2019-04-01    16574
2019-05-01    16574
2019-06-01    16574
2019-07-01    16574
2019-08-01    16574
2019-09-01    16574
2019-10-01    16574
2019-11-01    16574
2019-12-01    16574
2020-01-01    16574
2020-02-01    16574
2020-03-01    16574
2020-04-01    16574
2020-05-01    16574
2020-06-01    16574
2020-07-01    16574
2020-08-01    16574
2020-09-01    16574
2020-10-01    16574
2020-11-01    16574
2020-12-01    16574
2021-01-01    16574
2021-02-01    16574
2021-03-01    16574
2021-04-01    16574
2021-05-01    16574
2021-06-01    16574
2021-07-01    16574
2021-08-01    16574
2021-09-01    16574
2021-10-01    16574
2021-11-01    16574
2021-12-01    16574
Name: count, dtype: int64

Since the Dataset is from 2019 to 2021, the dates should be between 01/01/2019 and 12/31/2021. The unique values of the column confirm this.

## 2.3 Logic Checks <a id="logic-checks"></a>


In this Section we perform logic checks on the data to ensure that the values in the columns make sense in relation to each other. We check for inconsistencies and anomalies that could indicate errors in the data.

### 2.3.1 Logic Check - DistanceKM and NumFlights <a id="logic-distancekm-vs-numflights"></a>


We want to check if there are any entries where NumFlights is 0 but DistanceKM is greater than 0. This would be a logical inconsistency, as it would imply that the customer traveled a distance without taking any flights.

In [49]:
# Check flights for DistanceKM and NumFlights. If NumFlights is 0, DistanceKM should also be 0 and vice versa.
# Check first how many entries have NumFlights = 0 but DistanceKM > 0
inconsistent_distance = flights[(flights["NumFlights"] == 0) & (flights["DistanceKM"] > 0)]
print("Entries with NumFlights = 0 but DistanceKM > 0:", inconsistent_distance.shape[0])
display(inconsistent_distance.head(3))

# Check first how many entries have DistanceKM = 0 but NumFlights > 0
inconsistent_numflights = flights[(flights["DistanceKM"] == 0) & (flights["NumFlights"] > 0)]
print("Entries with DistanceKM = 0 but NumFlights > 0:", inconsistent_numflights.shape[0])
display(inconsistent_numflights.head(3))

Entries with NumFlights = 0 but DistanceKM > 0: 11575


,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
19,261109,2021,12,2021-12-01,0,0,13736.0,1373.0,0.0,0.0
93,817609,2021,12,2021-12-01,0,0,23775.0,2377.0,0.0,0.0
96,192600,2021,12,2021-12-01,0,0,5119.0,511.0,0.0,0.0


Entries with DistanceKM = 0 but NumFlights > 0: 0


,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed


For our Clustering we need better data quality. So we will remove these inconsistent entries from the dataset. We will put NumFlights to NaN, where it was 0 and DistanceKM > 0 and will impute

In [50]:
# Delete NumFlight entries where it is 0 but DistanceKM > 0
flights.loc[(flights["NumFlights"] == 0) & (flights["DistanceKM"] > 0), "NumFlights"] = np.nan

In [51]:
# Check for NaN values in NumFlights after the deletion
num_nan_numflights = flights["NumFlights"].isna().sum()
print("Number of NaN values in NumFlights after deletion:", num_nan_numflights)

Number of NaN values in NumFlights after deletion: 11575


For the Imputation we will use the nearest neighbor imputer, based on the DistanceKM feature to fill in the missing NumFlights values. This approach assumes that customers who traveled similar distances are likely to have taken a similar number of flights.

In [52]:
# KNN-Imputation of NumFlights based on DistanceKM
imputer = KNNImputer(n_neighbors=5)
flights[['NumFlights', 'DistanceKM']] = imputer.fit_transform(flights[['NumFlights', 'DistanceKM']])
# Check for NaN values in NumFlights after imputation
num_nan_numflights_after_imputation = flights["NumFlights"].isna().sum()
print("Number of NaN values in NumFlights after imputation:", num_nan_numflights_after_imputation)

Number of NaN values in NumFlights after imputation: 0


### 2.3.2 Logic Check - NumFlights and FlightCompanions <a id="logic-numflights-vs-flightcompanions"></a>


Customers who have companions on flights should have at least one flight. And the number of flights with companions should not exceed the total number of flights. We want to check if there are any entries where NumFlightsWithCompanions is greater than 0 but NumFlights is 0. Furthermore we check if there are any entries where NumFlightsWithCompanions is greater than NumFlights. If we check for the second Condition, then this would include the first condition This would be a logical inconsistency, as it would imply that the customer has companions on flights without taking any flights themselves. 

In [53]:
# Check for entries where NumFlightsWithCompanions > NumFlights
inconsistent_companions = flights[(flights["NumFlightsWithCompanions"] > flights["NumFlights"])]
print("Entries with NumFlightsWithCompanions > NumFlights:", inconsistent_companions.shape[0])
display(inconsistent_companions.head(3))

Entries with NumFlightsWithCompanions > NumFlights: 0


,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed


# 3. Feature Engineering  <a class="anchor" id="feature-engineering"></a>

In this section, we will perform feature engineering to enhance our dataset for better analysis. This includes adding new features based on existing data.

## 3.1 Active Customers <a id="active-customers"></a>


We create a new column "is_active" in the customers dataframe to indicate whether a customer is active or not based on the CancellationDate column. If the CancellationDate is missing (NaT), we assume the customer is still active and set is_active to 1. Otherwise, we set it to 0. We look at the distribution of the newly created column to see how many active and inactive customers we have.

In [54]:
customers = customers.copy()   # <-- crucial to avoid SettingWithCopyWarning

# New column "IsActive": 1 if CancellationDate is NaN, else 0
customers["IsActive"] = customers["CancellationDate"].isna().astype(int)
# percentage distribution
customers["IsActive"].value_counts(normalize=True).round(3) * 100

IsActive
1    86.4
0    13.6
Name: proportion, dtype: float64

## 3.2 Tenure Days  <a id="tenure-days"></a>


Tenure Days is calculated by the difference between EnrollmentDate and CancellationDate. If the CancellationDate is missing, we assume the customer is still active and use the last date in the dataset (31/12/2021) for the calculation.

In [55]:
customers = customers.copy()   # <-- crucial to avoid SettingWithCopyWarning

today = pd.Timestamp('2021-12-31').normalize()

# Safe conversions
customers['EnrollmentDateOpening'] = pd.to_datetime(customers['EnrollmentDateOpening'], errors='coerce')
customers['CancellationDate']      = pd.to_datetime(customers['CancellationDate'], errors='coerce')

# EndDate: CancellationDate or today
customers['EndDate'] = customers['CancellationDate'].fillna(today)

# Tenure in days
customers['CustomerTenureDays'] = (customers['EndDate'] - customers['EnrollmentDateOpening']).dt.days

# Drop temp column (avoid inplace on possible views)
customers = customers.drop(columns=['EndDate'])

# Head of the updated customers dataframe with the columns related to tenure
customers[['EnrollmentDateOpening', 'CancellationDate', 'CustomerTenureDays']].head()


,EnrollmentDateOpening,CancellationDate,CustomerTenureDays
0,2019-02-15,NaT,1050
1,2019-03-09,NaT,1028
2,2017-07-14,2021-01-08,1274
3,2016-02-17,NaT,2144
4,2017-10-25,NaT,1528


## 3.3 Flight Metrics per Customer  <a id="flight-metrics-per-customer"></a>


- We will add some flight metrics grouped by customer into the customers dataframe. First we calculate them in a new dataframe and then we merge them into the customers dataframe.

### 3.3.1 Flights Aggregated Metrics  <a id="flights-aggregated-metrics"></a>


In [56]:
# Basic flight metrics per customer
flight_aggs = flights.groupby('Loyalty#').agg(
    # Aggregated Values
    total_flights=('NumFlights', 'sum'),
    total_flights_with_companions=('NumFlightsWithCompanions', 'sum'), 
    total_distance=('DistanceKM', 'sum'),
    total_points_accumulated=('PointsAccumulated', 'sum'),
    total_points_redeemed=('PointsRedeemed', 'sum'),
    total_cost_redeemed=('DollarCostPointsRedeemed', 'sum'),
    average_distance_per_flight=('DistanceKM', lambda x: round(x.sum() / x.count(), 2) if x.count() > 0 else 0)
).reset_index()

print("Flight aggregations summary:")

# Add derived metrics
flight_aggs['points_redemption_ratio'] = (
    flight_aggs['total_points_redeemed'] / 
    flight_aggs['total_points_accumulated'].replace(0, np.nan)
).fillna(0)

flight_aggs['companion_flight_ratio'] = (
    flight_aggs['total_flights_with_companions'] / 
    flight_aggs['total_flights']
).fillna(0)

display(flight_aggs.head())


Flight aggregations summary:


,Loyalty#,total_flights,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_cost_redeemed,average_distance_per_flight,points_redemption_ratio,companion_flight_ratio
0,100018,225.0,45,530230.0,53014.30,20562.8,201.9,14728.61,0.387873,0.200000
1,100102,245.0,55,339114.6,33903.96,18760.6,186.2,9419.85,0.553345,0.224490
2,100140,219.8,51,432030.8,43192.58,4896.0,48.0,12000.86,0.113353,0.232029
3,100214,128.4,17,364601.7,36453.77,12908.6,127.3,10127.83,0.354109,0.132399
4,100272,199.2,50,429630.5,42953.25,10891.4,107.0,11934.18,0.253564,0.251004


### 3.3.2 Merge Flights Aggregated with Customers  <a id="merge-flights-aggregated-with-customers"></a>


In [57]:
# Add the aggregated data from flights to the customer data
customers = customers.merge(flight_aggs, on='Loyalty#', how='left')

# Show the first 5 rows of the updated customers dataframe
display(customers.head())

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,...,CustomerTenureDays,total_flights,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_cost_redeemed,average_distance_per_flight,points_redemption_ratio,companion_flight_ratio
0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,...,1050,201.4,53,507054.9,50699.39,13517.9,134.6,14084.86,0.266628,0.263158
1,549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,...,1028,289.0,30,426827.4,42672.54,22457.8,221.4,11856.32,0.526282,0.103806
2,429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,...,1274,140.6,37,238376.1,23832.41,5479.6,53.2,6621.56,0.229922,0.263158
3,608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,...,2144,198.2,55,386029.3,38595.63,16331.5,162.2,10723.04,0.423144,0.277497
4,530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,...,1528,190.8,59,369242.6,36916.56,0.0,0.0,10256.74,0.000000,0.309224


## 3.4 Season to flights dataset <a id="season-to-flights-dataset"></a>


In this Section we add the Column "Season" to the flights dataset based on the Month column. We define the seasons as follows:
- Winter: December, January, February
- Spring: March, April, May
- Summer: June, July, August
- Autumn: September, October, November


We will use this in our Behavioral Segmentation analysis later.

In [58]:
# Add Column "Season" to flights dataset based on the Month column
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    else:
        return np.nan
    
flights['Month'] = flights['YearMonthDate'].dt.month
flights['Season'] = flights['Month'].apply(month_to_season) 
flights[['YearMonthDate', 'Month', 'Season']].head()

# Show the first 5 rows of the updated flights dataframe
display(flights.head())

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed,Season
0,413052,2021,12,2021-12-01,2.0,2,9384.0,938.0,0.0,0.0,Winter
1,464105,2021,12,2021-12-01,0.0,0,0.0,0.0,0.0,0.0,Winter
2,681785,2021,12,2021-12-01,10.0,3,14745.0,1474.0,0.0,0.0,Winter
3,185013,2021,12,2021-12-01,16.0,4,26311.0,2631.0,3213.0,32.0,Winter
4,216596,2021,12,2021-12-01,9.0,0,19275.0,1927.0,0.0,0.0,Winter


## 3.5 Logic Checks of the new columns <a id="logic-checks-new-columns"></a>


After Feature Engineering, we perform logic checks on the newly created columns to ensure their validity. We already did that in Section 2.3, but we will repeat the checks here to confirm that the new features are consistent with the existing data.

### 3.5.1 Logic Check - Customer Lifetime Value and Zero Flights <a id="logic-clv-and-zero-flights"></a>


We should check if there are any entries where Customer Lifetime Value is greater than 0 but NumFlights is 0. This would be a logical inconsistency, as it would imply that the customer has a lifetime value without taking any flights. So far we dont have a lot of information, how the Customer Lifetime Value is calculated. But it is reasonable to assume, that a customer who has not taken any flights should not have any lifetime value.

In [59]:
# Check for entries where Customer Lifetime Value > 0 but NumFlights = 0
inconsistent_clv = customers[(customers["Customer Lifetime Value"] > 0) & (customers["total_flights"] == 0)]
print("Entries with Customer Lifetime Value > 0 but NumFlights = 0:", inconsistent_clv.shape[0])
display(inconsistent_clv.head(3))

Entries with Customer Lifetime Value > 0 but NumFlights = 0: 1469


,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,...,CustomerTenureDays,total_flights,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_cost_redeemed,average_distance_per_flight,points_redemption_ratio,companion_flight_ratio
30,201574,Trudy,Roscoe,Trudy Roscoe,Canada,Ontario,Ottawa,45.421532,-75.697189,K1F 2R2,...,244,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38,834891,Wendell,Besley,Wendell Besley,Canada,British Columbia,Vancouver,49.282730,-123.120740,V5R 1W3,...,41,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
66,329382,Ngoc,Dubuisson,Ngoc Dubuisson,Canada,Quebec,Montreal,45.501690,-73.567253,H2T 9K8,...,2318,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We have found some inconsistent entries. We will try to fix them with imputation. First we need to put the NumFlights entries to NaN, where they are 0 and Customer Lifetime Value > 0. 

In [60]:
# NumFlights to NaN where Customer Lifetime Value > 0 but NumFlights = 0
customers.loc[(customers["Customer Lifetime Value"] > 0) & (customers["total_flights"] == 0), "total_flights"] = np.nan
# Check for NaN values in NumFlights after the deletion
num_nan_numflights = customers["total_flights"].isna().sum()
print("Number of NaN values in NumFlights after deletion:", num_nan_numflights)

Number of NaN values in NumFlights after deletion: 1469


Then we can impute them based on the Customer Lifetime Value feature. This approach assumes that customers with similar lifetime values are likely to have taken a similar number of flights. We use KNN-Imputation for this.

In [61]:
# Imputation of NumFlights based on Customer Lifetime Value
imputer = KNNImputer(n_neighbors=5)
customers[['total_flights', 'Customer Lifetime Value']] = imputer.fit_transform(customers[['total_flights', 'Customer Lifetime Value']])
# Check for NaN values in NumFlights after imputation
num_nan_numflights_after_imputation = customers["total_flights"].isna().sum()
print("Number of NaN values in NumFlights after imputation:", num_nan_numflights_after_imputation)

Number of NaN values in NumFlights after imputation: 0


### 3.5.2 Logic Check - AverageDistanceKM > 13,000 KM <a id="logic-avg-distancekm-over-13000-km"></a>


We want to check if there are any entries where a single flight is greater than 13,000 KM. DistanceKM is accumulated, but the average distance per flight should not exceed this value. The maximum distance between any two points in Canada is approximately 13,000 KM. If there are entries with AverageDistanceKM greater than this value, it would indicate a potential data error.

In [62]:
# Maximum possible distance for a single flight (approx. Vancouver → Singapore)
max_km_per_flight = 13000  

# Check for entries where average_distance_per_flight > 13,000 KM
inconsistent_distance = customers[customers["average_distance_per_flight"] > max_km_per_flight]
print("Entries with average_distance_per_flight > 13,000 KM:", inconsistent_distance.shape[0])
display(inconsistent_distance.head(3))

Entries with average_distance_per_flight > 13,000 KM: 1773


,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,...,CustomerTenureDays,total_flights,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_cost_redeemed,average_distance_per_flight,points_redemption_ratio,companion_flight_ratio
0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,...,1050,201.4,53,507054.9,50699.39,13517.9,134.6,14084.86,0.266628,0.263158
5,927943,Hue,Sellner,Hue Sellner,Canada,Ontario,Toronto,43.653225,-79.383186,P5S 6R4,...,1666,200.0,32,519316.7,51924.87,21597.7,214.4,14425.46,0.415941,0.160000
12,988178,Andre,Cotugno,Andre Cotugno,Canada,Quebec,Montreal,45.501690,-73.567253,H4G 3T4,...,1905,309.0,86,557976.5,55788.05,14951.1,147.4,15499.35,0.267998,0.278317


We assume, that the total_flights are not correct, because total_distance and total_points_accumulated are  similar, which makes sense.
That is why we will put the total_flights to NaN, where the AverageDistanceKM is greater than 13,000 KM and will impute them based on the total_distance feature. This approach assumes that customers who traveled similar distances are likely to have taken a similar number of flights. We use KNN-Imputation for this. 


In [63]:
# total_flights to NaN where average_distance_per_flight > 13,000 KM
customers.loc[customers["average_distance_per_flight"] > max_km_per_flight, "total_flights"] = np.nan
# Check for NaN values in total_flights after the deletion
num_nan_total_flights = customers["total_flights"].isna().sum()
print("Number of NaN values in total_flights after deletion:", num_nan_total_flights)

Number of NaN values in total_flights after deletion: 1773


In [64]:
# Impute total_flights based on total_distance with KNN-Imputer
customers[['total_flights', 'total_distance']] = imputer.fit_transform(customers[['total_flights', 'total_distance']])
# Check for NaN values in total_flights after imputation
num_nan_total_flights_after_imputation = customers["total_flights"].isna().sum()
print("Number of NaN values in total_flights after imputation:", num_nan_total_flights_after_imputation)

Number of NaN values in total_flights after imputation: 0


### 3.5.3 Logic Check - Negative Customer Tenure Days <a id="logic-negative-tenure-days"></a>


We want to check if there are entries, where the Customer Tenure Days is negative. This would indicate a data error, as the CancellationDate should always be after the EnrollmentDate.

In [65]:
# Check for entries where the Customer Tenure Days is negative
inconsistent_tenure = customers[customers["CustomerTenureDays"] < 0]
print("Entries with negative Customer Tenure Days:", inconsistent_tenure.shape[0])
display(inconsistent_tenure.head(3))

Entries with negative Customer Tenure Days: 199


,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,...,CustomerTenureDays,total_flights,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_cost_redeemed,average_distance_per_flight,points_redemption_ratio,companion_flight_ratio
71,357549,Elisha,Furna,Elisha Furna,Canada,British Columbia,Whistler,50.116322,-122.957360,V6T 1Y8,...,-854,184.2,0,0.0,0.0,0.0,0.0,0.00,0.000000,0.000000
76,265297,Ebonie,Radde,Ebonie Radde,Canada,Manitoba,Winnipeg,49.895138,-97.138374,R2C 0M5,...,-853,38.0,3,46838.0,4683.0,0.0,0.0,1301.06,0.000000,0.078947
172,845613,Jerald,Shiring,Jerald Shiring,Canada,Quebec,Montreal,45.501690,-73.567253,H2Y 4R4,...,-853,12.0,4,31562.0,3155.0,5754.0,57.0,876.72,1.823772,0.333333


In [66]:
# Show the entries with negative Customer Tenure Days and their EnrollmentDateOpening and CancellationDate
inconsistent_tenure[['EnrollmentDateOpening', 'CancellationDate', 'CustomerTenureDays']]    


,EnrollmentDateOpening,CancellationDate,CustomerTenureDays
71,2021-09-21,2019-05-21,-854
76,2021-07-17,2019-03-17,-853
172,2021-10-13,2019-06-13,-853
205,2021-05-11,2019-01-11,-851
298,2021-07-20,2019-03-20,-853
...,...,...,...
16191,2021-08-18,2019-04-18,-853
16199,2021-07-21,2019-03-21,-853
16228,2021-10-18,2019-06-18,-853
16310,2021-08-03,2019-04-03,-853


In [67]:
# Show the highest and lowest Value of EnrollemntDateOpening and CancellationDate
print("Earliest EnrollmentDateOpening:", customers['EnrollmentDateOpening'].min())
print("Latest EnrollmentDateOpening:", customers['EnrollmentDateOpening'].max())
print("Earliest CancellationDate:", customers['CancellationDate'].min())
print("Latest CancellationDate:", customers['CancellationDate'].max())

Earliest EnrollmentDateOpening: 2015-04-01 00:00:00
Latest EnrollmentDateOpening: 2021-12-30 00:00:00
Earliest CancellationDate: 2015-11-30 00:00:00
Latest CancellationDate: 2021-12-30 00:00:00


In [68]:
# Calculate the percentage of negative tenure entries
total_customers = customers.shape[0]
negative_tenure_count = inconsistent_tenure.shape[0]
negative_tenure_percentage = (negative_tenure_count / total_customers) * 100
print(f"Percentage of entries with negative Customer Tenure Days: {negative_tenure_percentage:.4f}%")

Percentage of entries with negative Customer Tenure Days: 1.2007%


We can see the reason behind the negative tenure days. Could be switch between enrollment and cancellation or data entry error for the CancellationDate. Since we cannot determine the correct values for these entries, we will remove them from the dataset to ensure data integrity. The percentage of affected entries is relatively low, so the impact on the overall analysis should be minimal. 

In [69]:
# Delete entries with negative Customer Tenure Days
customers = customers[customers["CustomerTenureDays"] >= 0]
# Check for remaining negative Customer Tenure Days entries
remaining_inconsistent_tenure = customers[customers["CustomerTenureDays"] < 0]
print("Remaining entries with negative Customer Tenure Days after deletion:", remaining_inconsistent_tenure.shape[0])

Remaining entries with negative Customer Tenure Days after deletion: 0


# 4. Feature Encoding   <a class="anchor" id="feature-encoding"></a>

We have some features that are categorical and need to be encoded for clustering. We will use One-Hot Encoding for the EnrollmentType and LoyaltyStatus features. These features have a small number of unique values, so One-Hot Encoding is appropriate.


In [70]:
# One-Hot Encoding of EnrollmentType and LoyaltyStatus features
#customers = pd.get_dummies(customers, columns=['EnrollmentType', 'LoyaltyStatus'], drop_first=True)

# Print all columns of customers after encoding which start with EnrollmentType_ or LoyaltyStatus_
print("Columns after One-Hot Encoding:")
print([col for col in customers.columns if col.startswith('EnrollmentType_') or col.startswith('LoyaltyStatus_')])

Columns after One-Hot Encoding:
[]


# Check the dataset and export for Clustering

Now that we have cleaned and aggregated the data, we will export the final datasets for clustering analysis in the next part of the project. We will save the cleaned customers and flights datasets as CSV files for further analysis.

In [71]:
# Check both datasets before exporting
print("Final Customers shape:", customers.shape)
print("Final Flights shape:", flights.shape)

Final Customers shape: (16375, 31)
Final Flights shape: (596664, 11)


In [72]:
# Check the columns of both datasets before exporting
print("Customers columns:", customers.columns.tolist())
print("Flights columns:", flights.columns.tolist())

Customers columns: ['Loyalty#', 'First Name', 'Last Name', 'Customer Name', 'Country', 'Province or State', 'City', 'Latitude', 'Longitude', 'Postal code', 'Gender', 'Education', 'Location Code', 'Income', 'Marital Status', 'LoyaltyStatus', 'EnrollmentDateOpening', 'CancellationDate', 'Customer Lifetime Value', 'EnrollmentType', 'IsActive', 'CustomerTenureDays', 'total_flights', 'total_flights_with_companions', 'total_distance', 'total_points_accumulated', 'total_points_redeemed', 'total_cost_redeemed', 'average_distance_per_flight', 'points_redemption_ratio', 'companion_flight_ratio']
Flights columns: ['Loyalty#', 'Year', 'Month', 'YearMonthDate', 'NumFlights', 'NumFlightsWithCompanions', 'DistanceKM', 'PointsAccumulated', 'PointsRedeemed', 'DollarCostPointsRedeemed', 'Season']


In [73]:
# Check for NaN Values in the datasets
print("Missing values in final Customers dataset:")
display(customers.isna().sum())
print("Missing values in final Flights dataset:")
display(flights.isna().sum())

Missing values in final Customers dataset:


Loyalty#                             0
First Name                           0
Last Name                            0
Customer Name                        0
Country                              0
Province or State                    0
City                                 0
Latitude                             0
Longitude                            0
Postal code                          0
Gender                               0
Education                            0
Location Code                        0
Income                               0
Marital Status                       0
LoyaltyStatus                        0
EnrollmentDateOpening                0
CancellationDate                 14329
Customer Lifetime Value              0
EnrollmentType                       0
IsActive                             0
CustomerTenureDays                   0
total_flights                        0
total_flights_with_companions        0
total_distance                       0
total_points_accumulated 

Missing values in final Flights dataset:


Loyalty#                    0
Year                        0
Month                       0
YearMonthDate               0
NumFlights                  0
NumFlightsWithCompanions    0
DistanceKM                  0
PointsAccumulated           0
PointsRedeemed              0
DollarCostPointsRedeemed    0
Season                      0
dtype: int64

In [74]:
# Export the cleaned and feature engineered datasets to CSV files 

output_data_dir = Path("../data/cleanAndFeatureEngineered")

output_data_dir.mkdir(parents=True, exist_ok=True)


customers.to_csv(output_data_dir / "DM_AIAI_CustomerDB_Cleaned_Featured.csv", index=False)
flights.to_csv(output_data_dir / "DM_AIAI_FlightsDB_Cleaned_Featured.csv", index=False)

